#1. CROP and 반전
- 그러나 이 과정은, easyOCR 에서는 필요 없으므로 향후 생략

In [ ]:
import cv2
import numpy as np
import os
import glob
from pathlib import Path

# ==========================================
# 1. 경로 설정 (사용자 수정 포인트)
# ==========================================
# 기존 piece 이미지들이 저장된 폴더 경로
BASE_DIR = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/4. 결과 출력 시각화/2작기 error/260421 piece crop_ex/VER2_Gemini/crop")

# 결과가 저장될 하부 폴더들
SUB_DIRS = ["B", "C", "D", "E"]
for sd in SUB_DIRS:
    (BASE_DIR / sd).mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. 핵심 처리 함수들
# ==========================================

def get_white_plate_crop(img):
    """이미지 내에서 가장 밝은 흰색 사각형(번호판)만 찾아 크롭함"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 1단계: 흰색 번호판 추출 (밝기 200 이상만 남김)
    _, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)

    # 2단계: 윤곽선 찾기
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return img  # 못 찾으면 원본 반환

    # 3단계: 가장 큰 영역(번호판) 선택
    c = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(c)

    # 4단계: 약간의 여유(5px)를 두고 크롭
    pad = 5
    sub_crop = img[max(0, y-pad):min(img.shape[0], y+h+pad),
                   max(0, x-pad):min(img.shape[1], x+w+pad)]
    return sub_crop

def process_ocr_pre(img_path, save_base):
    """B, C, D, E 단계별 처리 및 저장"""
    img = cv2.imread(str(img_path))
    if img is None: return

    # 0단계: 번호판만 정밀 타겟팅 (Sub-crop)
    target_img = get_white_plate_crop(img)

    # --- B단계: 3배 확대 (보간법 적용) ---
    img_b = cv2.resize(target_img, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)

    # --- C단계: 4배 확대 + 대비(Contrast) 증가 ---
    img_c_base = cv2.resize(target_img, None, fx=4, fy=4, interpolation=cv2.INTER_CUBIC)
    gray_c = cv2.cvtColor(img_c_base, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img_c = clahe.apply(gray_c)

    # --- D단계: 이진화(Threshold) - 흰 판 기준 ---
    # Otsu 알고리즘을 써서 자동으로 최적 밝기값 찾기
    _, img_d = cv2.threshold(img_c, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # --- E단계: 반전(Binary Inversion) - 숫자만 검게 ---
    img_e = cv2.bitwise_not(img_d)

    # 파일명 추출 및 저장
    fname = img_path.name
    cv2.imwrite(str(save_base / "B" / fname), img_b)
    cv2.imwrite(str(save_base / "C" / fname), img_c)
    cv2.imwrite(str(save_base / "D" / fname), img_d)
    cv2.imwrite(str(save_base / "E" / fname), img_e)

# ==========================================
# 3. 실행부 (대용량 처리 고려)
# ==========================================
image_files = list(BASE_DIR.glob("*.jpg"))
total_count = len(image_files)

print(f"[알림] 총 {total_count}개의 파일을 처리를 시작합니다.")

for i, f_path in enumerate(image_files):
    process_ocr_pre(f_path, BASE_DIR)

    if (i + 1) % 10 == 0:
        print(f"[{i + 1}/{total_count}] 처리 중...")

print("[완료] B, C, D, E 폴더에 모든 전처리 이미지가 저장되었습니다.")

[알림] 총 51개의 파일을 처리를 시작합니다.
[10/51] 처리 중...
[20/51] 처리 중...
[30/51] 처리 중...
[40/51] 처리 중...
[50/51] 처리 중...
[완료] B, C, D, E 폴더에 모든 전처리 이미지가 저장되었습니다.


#2. EASYOCR 로 50개 파일 돌려서 CSV 출력하기

In [2]:
!pip install torch torchvision torchaudio
!pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 32.2 MB/s eta 0:00:00


In [3]:
import easyocr
import cv2
from matplotlib import pyplot as plt

In [ ]:
import os

image_root = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/4. 결과 출력 시각화/2작기 error/260421 piece crop_ex/VER2_Gemini/crop"

# EasyOCR 리더 초기화 (한국어와 영어를 지원하도록 설정)
reader = easyocr.Reader(['ko', 'en'])

# 이미지 파일 목록 가져오기
image_files = [os.path.join(image_root, f) for f in os.listdir(image_root) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
image_files.sort() # 파일 순서를 일관성 있게 정렬

print(f"Processing first {min(5, len(image_files))} images from {image_root}")

# 첫 5개의 이미지에 대해 EasyOCR 수행
for i, image_path in enumerate(image_files[:5]):
    print(f"\n--- Processing image {i+1}: {os.path.basename(image_path)} ---")
    img = cv2.imread(image_path)

    if img is None:
        print(f"Error: Could not read image {image_path}")
        continue

    # 이미지에서 텍스트 인식
    results = reader.readtext(img)

    # 결과 출력
    if results:
        for (bbox, text, prob) in results:
            print(f"Detected: {text}, Confidence: {prob:.4f}")
    else:
        print("No text detected.")

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.1% CompleteProcessing first 5 images from /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/4. 결과 출력 시각화/2작기 error/260421 piece crop_ex/VER2_Gemini/crop

--- Processing image 1: bed01_20260318_131655_cam1_piece.jpg ---


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Detected: 3, Confidence: 0.6509

--- Processing image 2: bed01_20260318_164957_cam1_piece.jpg ---
Detected: 18, Confidence: 0.9996

--- Processing image 3: bed01_20260320_133410_cam1_piece.jpg ---
Detected: 93, Confidence: 0.5453

--- Processing image 4: bed01_20260325_164327_cam1_piece.jpg ---
Detected: 63, Confidence: 1.0000

--- Processing image 5: bed01_20260325_180755_cam1_piece.jpg ---
Detected: 11, Confidence: 0.9986


50장 전체에 대해 돌려보는 코드

In [ ]:
import easyocr
import pandas as pd
import os
import re

# 1. 경로 설정
image_root = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/4. 결과 출력 시각화/2작기 error/260421 piece crop_ex/VER2_Gemini/crop"
csv_save_path = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/4. 결과 출력 시각화/2작기 error/260421 piece crop_ex/VER2_Gemini/260422_bed_ocr_results_crop.csv"
OCR_PREPROCESS_VERSION = "A_crop_raw_v1_easyocr_en"
MIN_ANCHOR_CONFIDENCE = 0.80

# 2. EasyOCR 로드
reader = easyocr.Reader(['en'], gpu=True)

def parse_file_info(file_name):
    # 예: bed01_20260318_131655_cam1_piece.jpg
    match = re.search(r'bed(\d+)_(\d{8})_(\d{6})_cam1_piece', file_name)
    if not match:
        return None, None, None
    bed_num = int(match.group(1))
    date = match.group(2)
    time = match.group(3)
    return bed_num, date, time

def summarize_candidates(ocr_res):
    # 예: 93:0.9297 | 9:0.3120
    items = []
    for _, text, conf in ocr_res:
        items.append(f"{text}:{round(conf, 4)}")
    return " | ".join(items)

def plate_qc_label(detected_num, confidence, ocr_is_single_digit):
    if detected_num == -1:
        return "NO_OCR"
    if not (1 <= detected_num <= 95):
        return "OUT_OF_RANGE"
    if confidence < MIN_ANCHOR_CONFIDENCE:
        return "LOW_CONF"
    if ocr_is_single_digit:
        return "SINGLE_DIGIT_REVIEW"
    return "OK_ANCHOR"

def run_batch_ocr(folder_path):
    results = []

    # 폴더 내 이미지 파일 리스트 (50개)
    files = [
        f for f in os.listdir(folder_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        and '_piece' in f.lower()
        and 'montage' not in f.lower()
    ]
    files.sort()

    for file_name in files:
        full_path = os.path.join(folder_path, file_name)

        # [A] 파일명에서 saved_bed/date/time 추출
        bed_num, date, time = parse_file_info(file_name)

        # [B] OCR 추론
        ocr_res = reader.readtext(full_path, allowlist='0123456789')
        ocr_all_candidates = summarize_candidates(ocr_res)

        if ocr_res:
            # 첫 번째 검출 결과 사용 (텍스트, 신뢰도)
            detected_text = ocr_res[0][1]
            confidence = round(ocr_res[0][2], 4)

            # 읽어온 텍스트에서 숫자만 추출 (예: "4 3" -> 43 / "bed01" -> 1)
            num_only = re.sub(r'[^0-9]', '', detected_text)
            detected_num = int(num_only) if num_only else -1
        else:
            detected_text = "None"
            num_only = ""
            detected_num = -1
            confidence = 0

        # [C] 결과 비교 (True/False)
        is_match = (bed_num == detected_num)
        ocr_text_length = len(num_only)
        ocr_is_single_digit = (ocr_text_length == 1)
        anchor_candidate = (
            detected_num != -1
            and 1 <= detected_num <= 95
            and confidence >= MIN_ANCHOR_CONFIDENCE
        )
        plate_qc = plate_qc_label(detected_num, confidence, ocr_is_single_digit)

        # single digit은 3처럼 진짜일 수도 있지만, 81 -> 8처럼 일부 누락일 수도 있으므로
        # 최종 anchor에서는 보수적으로 제외하고 사람이 확인할 후보로 둔다.
        final_anchor_flag = (anchor_candidate and not ocr_is_single_digit)

        results.append({
            'image_name': file_name,
            'bed_num': bed_num,
            'date': date,
            'time': time,
            'detected_num': detected_num,
            'CONFIDENCE': confidence,
            'ocr_all_candidates': ocr_all_candidates,
            'ocr_preprocess_version': OCR_PREPROCESS_VERSION,
            'ocr_text_length': ocr_text_length,
            'ocr_is_single_digit': ocr_is_single_digit,
            'anchor_candidate': anchor_candidate,
            'plate_qc': plate_qc,
            'final_anchor_flag': final_anchor_flag,
            'bools': is_match
        })

    # 3. CSV 저장
    df = pd.DataFrame(results)
    df.to_csv(csv_save_path, index=False, encoding='utf-8-sig')
    print(f"✅ 분석 완료. 파일 저장됨: {csv_save_path}")
    return df

# 실행
final_df = run_batch_ocr(image_root)


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argume

✅ 분석 완료. 파일 저장됨: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/4. 결과 출력 시각화/2작기 error/260421 piece crop_ex/VER2_Gemini/260422_bed_ocr_results_crop.csv



현재 통찰:
1. EasyOCR은 A piece crop 원본만으로도 충분히 강하다.
   B/C/D/E 전처리는 Tesseract를 살리기 위한 보정 단계에 가까웠고, EasyOCR을 메인으로 쓰면 운영 파이프라인에서 생략 가능성이 크다.

2. Tesseract는 경량이지만 현장 이미지에서는 실패 대비 코드가 길어진다.
   원근 보정, 테두리 제거, 숫자 component 분리, threshold 조정, psm 비교, 예외 처리 등을 붙여야 한다.
   결과적으로 모델은 가볍지만 전체 파이프라인은 복잡해질 수 있다.

3. EasyOCR을 쓰면 모델은 하나 늘지만, 전체 흐름은 단순해진다.
   cam1 원본 -> 오른쪽 번호표 A crop -> EasyOCR -> detected_num/confidence -> real_bed_id 검토
   이 구조가 현재로서는 가장 현실적이다.

4. 이번 결과는 “bed naming mismatch”가 실제 시계열 붕괴의 큰 원인이라는 강한 증거다.
   saved_bed 기준으로 묶으면 서로 다른 실제 물리 bed가 한 시계열에 섞인다.
   예를 들어 saved_bed=bed01 안에 실제 3,18,93,63,11,73,71,81,32가 섞이는 식이다.
   이 상태에서는 bed-seg, lettuce-seg, slot, warp, scale, CNN을 아무리 고쳐도 시계열이 안정될 수 없다.

5. saved_bed는 물리 bed id가 아니라 촬영/저장 당시의 임시 index 또는 위치 index로 취급해야 한다.
   앞으로 최종 key는 saved_bed가 아니라 cam1 번호표에서 복구한 real_bed_id가 되어야 한다.

내일 2ND(5-6)에서 할 일:
1. 260422_bed_ocr_results_crop.csv에서 montage 행 제거
2. bools to human 기준으로 OCR 실패 2건 확인
3. confidence 낮은 케이스가 있는지 추가 확인
4. saved_bed별로 Human 값 변화 패턴 확인
5. saved_bed -> real_bed_id 관계가 날짜/시간 구간별로 offset인지, 회전 순서인지, 재시작 구간별로 바뀌는지 조사
6. 50장 테스트가 충분히 좋으므로, 다음 단계는 complete triplet 7,342개 중 cam1에 대해 같은 방식으로 OCR 확장
7. 확장 결과 CSV에는 최소한 다음 컬럼을 넣으면 됨:
   - image_name
   - saved_bed
   - saved_num
   - date
   - time
   - detected_num
   - confidence
   - valid_range_1_95
   - same_as_saved
   - manual_real_bed_id
   - note

주의:
- 전체 폴더를 돌릴 때 montage 파일이 섞이지 않도록 반드시 *_piece.jpg 또는 cam1 원본만 필터링할 것
- EasyOCR reader는 반복문 안에서 매번 만들지 말고, 시작 시 1회만 생성할 것
- 한국어 모델은 필요 없음. 숫자 번호표는 reader = easyocr.Reader(['en']) + allowlist='0123456789'로 충분함
- OCR 결과를 곧바로 final real_bed_id로 확정하지 말고, confidence와 날짜/시간 흐름, 컨베이어 순환 규칙으로 QC해야 함

현재 결론:
EasyOCR A crop 방식은 50장 테스트에서 96% 정확도를 보였다.
이 정도면 bed identity 복구의 메인 루트로 채택할 가치가 충분하다.
가장 중요한 발견은 OCR 성능보다도 saved_bed와 실제 물리 bed 번호가 크게 불일치한다는 사실이다.
따라서 다음 작업의 우선순위는 downstream 모델 개선이 아니라 real_bed_id manifest 복구다.

# 전체 7000장에 대해 돌리기

In [4]:
"""
Run EasyOCR on all cam1 piece crops and write a streaming CSV.

This script is intended for the full ~7k crop set.

Memory policy:
- do not keep OCR rows in a DataFrame/list
- write one CSV row immediately after each image
- use one EasyOCR reader by default to avoid loading the model multiple times

Recommended Colab run:
    !python 03_run_easyocr_piece_crop_7000.py --workers 1

CPU-only experimental parallel run:
    !python 03_run_easyocr_piece_crop_7000.py --gpu false --workers 2
"""

from __future__ import annotations

import argparse
import csv
import json
import os
import re
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from pathlib import Path


try:
    from tqdm import tqdm
except ImportError:
    class tqdm:  # type: ignore
        def __init__(self, total=None, desc=None, unit=None):
            self.count = 0

        def __enter__(self):
            return self

        def __exit__(self, exc_type, exc, tb):
            print(f"[INFO] processed: {self.count}")

        def update(self, n=1):
            self.count += n


# 1. 경로 설정
IMAGE_ROOT = Path(
    "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/OCR_Piece/2작기/1. CROP"
)
CSV_SAVE_PATH = Path(
    "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/OCR_Piece/2작기/260422_ocr_cam1.csv"
)

OCR_PREPROCESS_VERSION = "A_crop_raw_v1_easyocr_en"
MIN_ANCHOR_CONFIDENCE = 0.80
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

FILENAME_RE = re.compile(
    r"^(?P<saved_bed>bed(?P<saved_num>\d{1,3}))_"
    r"(?P<date>\d{8})_(?P<time>\d{6})_cam1_piece\."
    r"(jpg|jpeg|png|bmp|tif|tiff)$",
    re.IGNORECASE,
)

FIELDNAMES = [
    "image_name",
    "saved_bed",
    "saved_num",
    "date",
    "time",
    "plate_crop_path",
    "preprocess_version",
    "ocr_raw_text",
    "ocr_all_candidates",
    "ocr_best_text",
    "ocr_best_digits",
    "ocr_best_conf",
    "detected_num",
    "confidence",
    "ocr_text_len",
    "is_single_digit",
    "valid_range_1_95",
    "same_as_saved",
    "cyclic_offset_95",
    "anchor_candidate",
    "qc_status",
    "final_anchor_flag",
    "segment_id",
    "fill_method",
    "manual_visible_bed_num",
    "note",
]


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Run EasyOCR on all cam1 piece crops.")
    parser.add_argument("--image-root", type=Path, default=IMAGE_ROOT)
    parser.add_argument("--output-csv", type=Path, default=CSV_SAVE_PATH)
    parser.add_argument("--gpu", choices=["true", "false"], default="true")
    parser.add_argument("--workers", type=int, default=1)
    parser.add_argument("--max-in-flight", type=int, default=4)
    parser.add_argument("--limit", type=int, default=None)
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--min-anchor-confidence", type=float, default=MIN_ANCHOR_CONFIDENCE)
    parser.add_argument("--segment-date", default="20260403")
    args, _unknown = parser.parse_known_args()
    return args


def parse_file_info(file_name: str):
    m = FILENAME_RE.match(file_name)
    if not m:
        return None
    saved_num = int(m.group("saved_num"))
    return {
        "saved_bed": f"bed{saved_num:02d}",
        "saved_num": saved_num,
        "date": m.group("date"),
        "time": m.group("time"),
    }


def digits_only(text: str) -> str:
    return re.sub(r"[^0-9]", "", text or "")


def list_images(image_root: Path, limit: int | None = None) -> list[Path]:
    files = []
    for file_name in os.listdir(image_root):
        path = image_root / file_name
        if not path.is_file():
            continue
        if path.suffix.lower() not in IMAGE_EXTS:
            continue
        if "montage" in file_name.lower():
            continue
        if not FILENAME_RE.match(file_name):
            continue
        files.append(path)
    files.sort(key=lambda p: p.name)
    if limit is not None:
        files = files[:limit]
    return files


def load_done_images(output_csv: Path) -> set[str]:
    if not output_csv.exists():
        return set()
    done = set()
    with output_csv.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get("image_name"):
                done.add(row["image_name"])
    return done


def format_candidates(ocr_res) -> tuple[str, str, str, list[dict]]:
    rows = []
    raw_texts = []
    digit_texts = []
    best = None

    for bbox, text, conf in ocr_res:
        digits = digits_only(text)
        item = {
            "text": str(text),
            "digits": digits,
            "conf": round(float(conf), 6),
            "bbox": [[float(x), float(y)] for x, y in bbox],
        }
        rows.append(item)
        raw_texts.append(str(text))
        if digits:
            digit_texts.append(digits)
            if best is None or float(conf) > best["conf"]:
                best = item

    ocr_raw_text = " | ".join(raw_texts)
    ocr_all_candidates = json.dumps(rows, ensure_ascii=False)
    ocr_digit_candidates = " | ".join(digit_texts)
    return ocr_raw_text, ocr_digit_candidates, ocr_all_candidates, rows if best is None else [best] + rows


def qc_status_label(detected_num: int, confidence: float, text_len: int, min_conf: float) -> str:
    if detected_num == -1:
        return "NO_OCR"
    if not (1 <= detected_num <= 95):
        return "OUT_OF_RANGE"
    if confidence < min_conf:
        return "LOW_CONF"
    if text_len == 1:
        return "SINGLE_DIGIT_REVIEW"
    return "OK_ANCHOR"


def segment_id_for(date: str, segment_date: str) -> str:
    if not date:
        return ""
    return f"pre_{segment_date}" if date < segment_date else f"post_{segment_date}"


def cyclic_offset(saved_num: int, detected_num: int) -> str:
    if detected_num == -1 or not (1 <= detected_num <= 95):
        return ""
    return str((detected_num - saved_num) % 95)


def build_row(image_path: Path, ocr_res, min_conf: float, segment_date: str) -> dict:
    meta = parse_file_info(image_path.name)
    if meta is None:
        return {
            "image_name": image_path.name,
            "plate_crop_path": str(image_path),
            "qc_status": "BAD_FILENAME",
            "fill_method": "ocr_direct",
        }

    ocr_raw_text, ocr_digit_candidates, ocr_all_candidates, best_plus_all = format_candidates(ocr_res)
    best = best_plus_all[0] if best_plus_all and best_plus_all[0].get("digits") else None

    if best:
        detected_digits = best["digits"]
        detected_num = int(detected_digits) if detected_digits else -1
        confidence = float(best["conf"])
        best_text = best["text"]
    else:
        detected_digits = ""
        detected_num = -1
        confidence = 0.0
        best_text = ""

    text_len = len(detected_digits)
    is_single_digit = text_len == 1
    valid_range = detected_num != -1 and 1 <= detected_num <= 95
    same_as_saved = valid_range and detected_num == meta["saved_num"]
    offset = cyclic_offset(meta["saved_num"], detected_num)
    qc_status = qc_status_label(detected_num, confidence, text_len, min_conf)
    anchor_candidate = valid_range and confidence >= min_conf
    final_anchor_flag = qc_status == "OK_ANCHOR"

    return {
        "image_name": image_path.name,
        "saved_bed": meta["saved_bed"],
        "saved_num": meta["saved_num"],
        "date": meta["date"],
        "time": meta["time"],
        "plate_crop_path": str(image_path),
        "preprocess_version": OCR_PREPROCESS_VERSION,
        "ocr_raw_text": ocr_raw_text,
        "ocr_all_candidates": ocr_all_candidates,
        "ocr_best_text": best_text,
        "ocr_best_digits": detected_digits,
        "ocr_best_conf": round(confidence, 6),
        "detected_num": detected_num,
        "confidence": round(confidence, 6),
        "ocr_text_len": text_len,
        "is_single_digit": is_single_digit,
        "valid_range_1_95": valid_range,
        "same_as_saved": same_as_saved,
        "cyclic_offset_95": offset,
        "anchor_candidate": anchor_candidate,
        "qc_status": qc_status,
        "final_anchor_flag": final_anchor_flag,
        "segment_id": segment_id_for(meta["date"], segment_date),
        "fill_method": "ocr_direct",
        "manual_visible_bed_num": "",
        "note": "",
    }


def make_reader(gpu: bool):
    import easyocr

    return easyocr.Reader(["en"], gpu=gpu)


def run_single_reader(args, image_paths: list[Path]) -> None:
    reader = make_reader(args.gpu == "true")
    write_header = not (args.resume and args.output_csv.exists())
    mode = "a" if args.resume else "w"

    args.output_csv.parent.mkdir(parents=True, exist_ok=True)
    with args.output_csv.open(mode, encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if write_header:
            writer.writeheader()

        with tqdm(total=len(image_paths), desc="EasyOCR", unit="img") as pbar:
            for image_path in image_paths:
                try:
                    ocr_res = reader.readtext(str(image_path), allowlist="0123456789")
                    row = build_row(image_path, ocr_res, args.min_anchor_confidence, args.segment_date)
                except Exception as exc:
                    meta = parse_file_info(image_path.name) or {}
                    row = {name: "" for name in FIELDNAMES}
                    row.update({
                        "image_name": image_path.name,
                        "saved_bed": meta.get("saved_bed", ""),
                        "saved_num": meta.get("saved_num", ""),
                        "date": meta.get("date", ""),
                        "time": meta.get("time", ""),
                        "plate_crop_path": str(image_path),
                        "preprocess_version": OCR_PREPROCESS_VERSION,
                        "qc_status": "READ_ERROR",
                        "fill_method": "ocr_direct",
                        "note": repr(exc),
                    })
                writer.writerow(row)
                f.flush()
                pbar.update(1)


_WORKER_READER = None
_WORKER_MIN_CONF = MIN_ANCHOR_CONFIDENCE
_WORKER_SEGMENT_DATE = "20260403"


def init_worker(gpu: bool, min_conf: float, segment_date: str):
    global _WORKER_READER, _WORKER_MIN_CONF, _WORKER_SEGMENT_DATE
    _WORKER_READER = make_reader(gpu)
    _WORKER_MIN_CONF = min_conf
    _WORKER_SEGMENT_DATE = segment_date


def worker_ocr(image_path: Path) -> dict:
    try:
        ocr_res = _WORKER_READER.readtext(str(image_path), allowlist="0123456789")
        return build_row(image_path, ocr_res, _WORKER_MIN_CONF, _WORKER_SEGMENT_DATE)
    except Exception as exc:
        meta = parse_file_info(image_path.name) or {}
        row = {name: "" for name in FIELDNAMES}
        row.update({
            "image_name": image_path.name,
            "saved_bed": meta.get("saved_bed", ""),
            "saved_num": meta.get("saved_num", ""),
            "date": meta.get("date", ""),
            "time": meta.get("time", ""),
            "plate_crop_path": str(image_path),
            "preprocess_version": OCR_PREPROCESS_VERSION,
            "qc_status": "READ_ERROR",
            "fill_method": "ocr_direct",
            "note": repr(exc),
        })
        return row


def run_parallel(args, image_paths: list[Path]) -> None:
    write_header = not (args.resume and args.output_csv.exists())
    mode = "a" if args.resume else "w"
    args.output_csv.parent.mkdir(parents=True, exist_ok=True)
    max_in_flight = max(args.workers, args.max_in_flight)

    def submit_next(executor, iterator, futures):
        while len(futures) < max_in_flight:
            try:
                image_path = next(iterator)
            except StopIteration:
                break
            futures.add(executor.submit(worker_ocr, image_path))

    with args.output_csv.open(mode, encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if write_header:
            writer.writeheader()

        with ThreadPoolExecutor(
            max_workers=args.workers,
            initializer=init_worker,
            initargs=(args.gpu == "true", args.min_anchor_confidence, args.segment_date),
        ) as executor:
            iterator = iter(image_paths)
            futures = set()
            submit_next(executor, iterator, futures)
            with tqdm(total=len(image_paths), desc="EasyOCR", unit="img") as pbar:
                while futures:
                    done, futures = wait(futures, return_when=FIRST_COMPLETED)
                    for future in done:
                        writer.writerow(future.result())
                        pbar.update(1)
                    f.flush()
                    submit_next(executor, iterator, futures)


def main() -> None:
    args = parse_args()
    images = list_images(args.image_root, args.limit)
    if args.resume:
        done = load_done_images(args.output_csv)
        images = [p for p in images if p.name not in done]
        print(f"[INFO] resume=True, already done: {len(done)}")

    print(f"[INFO] image_root     = {args.image_root}")
    print(f"[INFO] output_csv     = {args.output_csv}")
    print(f"[INFO] images to OCR  = {len(images)}")
    print(f"[INFO] gpu            = {args.gpu}")
    print(f"[INFO] workers        = {args.workers}")
    print(f"[INFO] segment_date   = {args.segment_date}")

    if not images:
        print("[INFO] No images to process.")
        return

    if args.workers <= 1:
        run_single_reader(args, images)
    else:
        print("[WARN] Parallel EasyOCR loads one reader per worker. Use workers=1 on GPU if RAM is tight.")
        run_parallel(args, images)

    print(f"[DONE] wrote: {args.output_csv}")


if __name__ == "__main__":
    main()


[INFO] image_root     = /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/OCR_Piece/2작기/1. CROP
[INFO] output_csv     = /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/OCR_Piece/2작기/260422_ocr_cam1.csv
[INFO] images to OCR  = 7352
[INFO] gpu            = true
[INFO] workers        = 1
[INFO] segment_date   = 20260403
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

EasyOCR: 100%|██████████| 7352/7352 [1:30:33<00:00,  1.35img/s]

[DONE] wrote: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/OCR_Piece/2작기/260422_ocr_cam1.csv
